<a href="https://colab.research.google.com/github/ashokmulchandani/CUDA-GPU-Colab-Computer-Vision-Project-Ashok-1/blob/main/1_Ashok_CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Check NVIDIA GPU
!nvidia-smi


Tue May 19 02:56:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2: Check CUDA compiler version
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
# Cell 3: Check GPU details
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


In [4]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv


name, compute_cap, memory.total [MiB]
Tesla T4, 7.5, 15360 MiB


In [7]:
!nvcc device_query.cu -o device_query && ./device_query


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Device: Tesla T4
Compute Capability: 7.5
SMs (Streaming Multiprocessors): 40
Max Threads per SM: 1024
Max Threads per Block: 1024
Warp Size: 32
Total Global Memory: 15.64 GB
Shared Memory per Block: 48 KB
Clock Rate: 1.59 GHz
Memory Clock: 5.00 GHz
Memory Bus Width: 256 bits


In [8]:
%%writefile hello_world.cu
#include <stdio.h>

// __global__ = this function runs on GPU, called from CPU
__global__ void hello() {
    printf("Hello from GPU! Thread %d in Block %d\n", threadIdx.x, blockIdx.x);
}

int main() {
    // Launch kernel: <<<num_blocks, threads_per_block>>>
    hello<<<2, 4>>>();  // 2 blocks, 4 threads each = 8 threads total

    // Wait for GPU to finish before CPU continues
    cudaDeviceSynchronize();

    printf("Hello from CPU!\n");
    return 0;
}


Writing hello_world.cu


In [9]:
!nvcc hello_world.cu -o hello_world && ./hello_world


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Hello from GPU! Thread 0 in Block 0
Hello from GPU! Thread 1 in Block 0
Hello from GPU! Thread 2 in Block 0
Hello from GPU! Thread 3 in Block 0
Hello from GPU! Thread 0 in Block 1
Hello from GPU! Thread 1 in Block 1
Hello from GPU! Thread 2 in Block 1
Hello from GPU! Thread 3 in Block 1
Hello from CPU!


In [1]:
%%writefile hello_world2.cu
#include <stdio.h>

__global__ void hello() {
    // Calculate unique global thread ID
    int globalId = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Thread %d (Block %d, Local %d)\n", globalId, blockIdx.x, threadIdx.x);
}

int main() {
    hello<<<4, 8>>>();  // 4 blocks × 8 threads = 32 threads (1 full warp!)
    cudaDeviceSynchronize();
    printf("Total threads launched: 32\n");
    return 0;
}


Writing hello_world2.cu


In [2]:
!nvcc hello_world2.cu -o hello_world2 && ./hello_world2


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Thread 16 (Block 2, Local 0)
Thread 17 (Block 2, Local 1)
Thread 18 (Block 2, Local 2)
Thread 19 (Block 2, Local 3)
Thread 20 (Block 2, Local 4)
Thread 21 (Block 2, Local 5)
Thread 22 (Block 2, Local 6)
Thread 23 (Block 2, Local 7)
Thread 8 (Block 1, Local 0)
Thread 9 (Block 1, Local 1)
Thread 10 (Block 1, Local 2)
Thread 11 (Block 1, Local 3)
Thread 12 (Block 1, Local 4)
Thread 13 (Block 1, Local 5)
Thread 14 (Block 1, Local 6)
Thread 15 (Block 1, Local 7)
Thread 24 (Block 3, Local 0)
Thread 25 (Block 3, Local 1)
Thread 26 (Block 3, Local 2)
Thread 27 (Block 3, Local 3)
Thread 28 (Block 3, Local 4)
Thread 29 (Block 3, Local 5)
Thread 30 (Block 3, Local 6)
Thread 31 (Block 3, Local 7)
Thread 0 (Block 0, Local 0)
Thread 1 (Block 0, Local 1)
Thread 2 (Block 0, Local 2)
Thread 3 (Block 0, Local 3)
Thread 

In [4]:
%%writefile vector_add.cu
#include <stdio.h>

// GPU kernel: each thread adds one element
__global__ void vectorAdd(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {          // guard: don't go out of bounds
        c[i] = a[i] + b[i];
    }
}

int main() {
    int n = 10;  // small array for now
    int size = n * sizeof(int);

    // --- CPU (Host) arrays ---
    int h_a[] = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10};
    int h_b[] = {10, 20, 30, 40, 50, 60, 70, 80, 90, 100};
    int h_c[10];  // result

    // --- GPU (Device) arrays ---
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);    // allocate on GPU
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // --- Copy data: CPU → GPU ---
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // --- Launch kernel ---
    int threadsPerBlock = 4;
    int blocks = (n + threadsPerBlock - 1) / threadsPerBlock;  // ceiling division
    printf("Launching %d blocks × %d threads = %d threads\n", blocks, threadsPerBlock, blocks * threadsPerBlock);

    vectorAdd<<<blocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    // --- Copy result: GPU → CPU ---
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // --- Print results ---
    for (int i = 0; i < n; i++) {
        printf("%d + %d = %d\n", h_a[i], h_b[i], h_c[i]);
    }

    // --- Free GPU memory ---
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Writing vector_add.cu


In [5]:
!nvcc vector_add.cu -o vector_add && ./vector_add


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Launching 3 blocks × 4 threads = 12 threads
1 + 10 = 11
2 + 20 = 22
3 + 30 = 33
4 + 40 = 44
5 + 50 = 55
6 + 60 = 66
7 + 70 = 77
8 + 80 = 88
9 + 90 = 99
10 + 100 = 110


In [1]:
%%writefile grid_2d.cu
#include <stdio.h>

__global__ void show2D() {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    printf("Thread at (row=%d, col=%d) — Block(%d,%d) Local(%d,%d)\n",
           row, col, blockIdx.y, blockIdx.x, threadIdx.y, threadIdx.x);
}

int main() {
    // 2D grid: 2×2 blocks, each block has 3×3 threads
    dim3 blocks(2, 2);      // 2 blocks in x, 2 blocks in y
    dim3 threads(3, 3);     // 3 threads in x, 3 threads in y per block

    printf("Grid: 2x2 blocks, each 3x3 threads\n");
    printf("Total threads: %d\n", 2*2*3*3);
    printf("Covers a 6x6 area (2*3 = 6 in each dimension)\n\n");

    show2D<<<blocks, threads>>>();
    cudaDeviceSynchronize();

    return 0;
}


Writing grid_2d.cu


In [2]:
!nvcc grid_2d.cu -o grid_2d && ./grid_2d


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Grid: 2x2 blocks, each 3x3 threads
Total threads: 36
Covers a 6x6 area (2*3 = 6 in each dimension)

Thread at (row=3, col=0) — Block(1,0) Local(0,0)
Thread at (row=3, col=1) — Block(1,0) Local(0,1)
Thread at (row=3, col=2) — Block(1,0) Local(0,2)
Thread at (row=4, col=0) — Block(1,0) Local(1,0)
Thread at (row=4, col=1) — Block(1,0) Local(1,1)
Thread at (row=4, col=2) — Block(1,0) Local(1,2)
Thread at (row=5, col=0) — Block(1,0) Local(2,0)
Thread at (row=5, col=1) — Block(1,0) Local(2,1)
Thread at (row=5, col=2) — Block(1,0) Local(2,2)
Thread at (row=0, col=0) — Block(0,0) Local(0,0)
Thread at (row=0, col=1) — Block(0,0) Local(0,1)
Thread at (row=0, col=2) — Block(0,0) Local(0,2)
Thread at (row=1, col=0) — Block(0,0) Local(1,0)
Thread at (row=1, col=1) — Block(0,0) Local(1,1)
Thread at (row=1, col=2) — 

In [3]:
%%writefile memory_demo.cu
#include <stdio.h>

__global__ void doubleValues(float *d_arr, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        d_arr[i] = d_arr[i] * 2.0f;  // modify data IN-PLACE on GPU
    }
}

int main() {
    int n = 5;
    int size = n * sizeof(float);  // 5 floats × 4 bytes = 20 bytes

    // ========== CPU (Host) Memory ==========
    float h_data[] = {1.0, 2.0, 3.0, 4.0, 5.0};

    printf("=== STEP 1: Data lives on CPU ===\n");
    printf("CPU address: %p\n", (void*)h_data);
    for (int i = 0; i < n; i++) printf("  h_data[%d] = %.1f\n", i, h_data[i]);

    // ========== GPU (Device) Memory ==========
    float *d_data;

    // Allocate memory on GPU (like malloc but on GPU)
    cudaMalloc(&d_data, size);
    printf("\n=== STEP 2: Allocated %d bytes on GPU ===\n", size);
    printf("GPU address: %p\n", (void*)d_data);
    printf("(CPU cannot read this address directly!)\n");

    // Copy data: CPU → GPU
    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);
    printf("\n=== STEP 3: Copied CPU → GPU ===\n");

    // Run kernel (doubles all values on GPU)
    doubleValues<<<1, n>>>(d_data, n);
    printf("=== STEP 4: Kernel ran (doubled values on GPU) ===\n");

    // Copy result: GPU → CPU
    cudaMemcpy(h_data, d_data, size, cudaMemcpyDeviceToHost);
    printf("\n=== STEP 5: Copied GPU → CPU ===\n");
    for (int i = 0; i < n; i++) printf("  h_data[%d] = %.1f\n", i, h_data[i]);

    // Free GPU memory
    cudaFree(d_data);
    printf("\n=== STEP 6: Freed GPU memory ===\n");

    return 0;
}


Writing memory_demo.cu


In [4]:
!nvcc memory_demo.cu -o memory_demo && ./memory_demo

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
=== STEP 1: Data lives on CPU ===
CPU address: 0x7ffde371fec0
  h_data[0] = 1.0
  h_data[1] = 2.0
  h_data[2] = 3.0
  h_data[3] = 4.0
  h_data[4] = 5.0

=== STEP 2: Allocated 20 bytes on GPU ===
GPU address: 0x7875c9000000
(CPU cannot read this address directly!)

=== STEP 3: Copied CPU → GPU ===
=== STEP 4: Kernel ran (doubled values on GPU) ===

=== STEP 5: Copied GPU → CPU ===
  h_data[0] = 2.0
  h_data[1] = 4.0
  h_data[2] = 6.0
  h_data[3] = 8.0
  h_data[4] = 10.0

=== STEP 6: Freed GPU memory ===


In [5]:
%%writefile pinned_memory.cu
#include <stdio.h>

int main() {
    int n = 10000000;  // 10 million floats = 40 MB
    int size = n * sizeof(float);
    float *d_data;
    cudaMalloc(&d_data, size);

    // ========== Timing events ==========
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms;

    // ========== TEST 1: Pageable memory (normal malloc) ==========
    float *h_pageable = (float*)malloc(size);
    for (int i = 0; i < n; i++) h_pageable[i] = 1.0f;

    cudaEventRecord(start);
    cudaMemcpy(d_data, h_pageable, size, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("Pageable memory:  CPU → GPU = %.2f ms  (%.2f GB/s)\n",
           ms, (size / 1e9) / (ms / 1000));

    // ========== TEST 2: Pinned memory (cudaMallocHost) ==========
    float *h_pinned;
    cudaMallocHost(&h_pinned, size);  // pinned = locked in RAM
    for (int i = 0; i < n; i++) h_pinned[i] = 1.0f;

    cudaEventRecord(start);
    cudaMemcpy(d_data, h_pinned, size, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("Pinned memory:    CPU → GPU = %.2f ms  (%.2f GB/s)\n",
           ms, (size / 1e9) / (ms / 1000));

    // Cleanup
    free(h_pageable);
    cudaFreeHost(h_pinned);
    cudaFree(d_data);

    return 0;
}


Writing pinned_memory.cu


In [6]:
!nvcc pinned_memory.cu -o pinned_memory && ./pinned_memory


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Pageable memory:  CPU → GPU = 8.22 ms  (4.87 GB/s)
Pinned memory:    CPU → GPU = 3.25 ms  (12.30 GB/s)


In [8]:
%%writefile unified_memory.cu
#include <stdio.h>

__global__ void doubleValues(float *data, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        data[i] = data[i] * 2.0f;
    }
}

int main() {
    int n = 5;

    // ========== OLD WAY (manual copies) ==========
    // float *h_data = (float*)malloc(size);
    // float *d_data; cudaMalloc(&d_data, size);
    // cudaMemcpy(d_data, h_data, size, HostToDevice);
    // kernel<<<...>>>(d_data);
    // cudaMemcpy(h_data, d_data, size, DeviceToHost);

    // ========== NEW WAY (unified memory) ==========
    float *data;
    cudaMallocManaged(&data, n * sizeof(float));  // ONE pointer, works on BOTH CPU and GPU!

    // CPU writes directly
    data[0] = 1.0; data[1] = 2.0; data[2] = 3.0; data[3] = 4.0; data[4] = 5.0;
    printf("Before kernel: ");
    for (int i = 0; i < n; i++) printf("%.1f ", data[i]);
    printf("\n");

    // GPU reads and writes the SAME pointer
    doubleValues<<<1, n>>>(data, n);
    cudaDeviceSynchronize();  // wait for GPU to finish

    // CPU reads directly (no cudaMemcpy needed!)
    printf("After kernel:  ");
    for (int i = 0; i < n; i++) printf("%.1f ", data[i]);
    printf("\n");

    cudaFree(data);
    return 0;
}


Writing unified_memory.cu


In [9]:
!nvcc unified_memory.cu -o unified_memory && ./unified_memory


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Before kernel: 1.0 2.0 3.0 4.0 5.0 
After kernel:  2.0 4.0 6.0 8.0 10.0 


In [10]:
%%writefile error_handling.cu
#include <stdio.h>

// Macro to check CUDA errors
#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            printf("CUDA Error at %s:%d - %s\n", __FILE__, __LINE__, \
                   cudaGetErrorString(err)); \
            exit(1); \
        } \
    } while(0)

__global__ void badKernel(float *data, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        data[i] = data[i] * 2.0f;
    }
}

int main() {
    int n = 5;
    int size = n * sizeof(float);
    float *d_data;

    // Good call — will succeed
    CUDA_CHECK(cudaMalloc(&d_data, size));
    printf("✓ cudaMalloc succeeded\n");

    // Check kernel errors (kernel errors are async!)
    badKernel<<<1, n>>>(d_data, n);
    CUDA_CHECK(cudaGetLastError());        // check launch errors
    CUDA_CHECK(cudaDeviceSynchronize());   // check execution errors
    printf("✓ Kernel succeeded\n");

    // Try to free same memory twice — will cause error
    CUDA_CHECK(cudaFree(d_data));
    printf("✓ First free succeeded\n");

    CUDA_CHECK(cudaFree(d_data));  // ERROR! Already freed
    printf("This won't print\n");

    return 0;
}


Writing error_handling.cu


In [11]:
!nvcc error_handling.cu -o error_handling && ./error_handling


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
✓ cudaMalloc succeeded
✓ Kernel succeeded
✓ First free succeeded
CUDA Error at error_handling.cu:40 - invalid argument


In [12]:
%%writefile timing.cu
#include <stdio.h>

__global__ void vectorAdd(float *a, float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    int n = 10000000;  // 10 million elements
    int size = n * sizeof(float);

    // Allocate unified memory (simpler for this demo)
    float *a, *b, *c;
    cudaMallocManaged(&a, size);
    cudaMallocManaged(&b, size);
    cudaMallocManaged(&c, size);

    // Initialize data
    for (int i = 0; i < n; i++) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    // Create timing events
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Launch config
    int threadsPerBlock = 256;
    int blocks = (n + threadsPerBlock - 1) / threadsPerBlock;

    // Time the kernel
    cudaEventRecord(start);
    vectorAdd<<<blocks, threadsPerBlock>>>(a, b, c, n);
    cudaEventRecord(stop);

    // Wait and calculate time
    cudaEventSynchronize(stop);
    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    // Verify result
    printf("c[0] = %.1f (expected 3.0)\n", c[0]);
    printf("c[9999999] = %.1f (expected 3.0)\n", c[9999999]);
    printf("\n");
    printf("Vector size: %d elements (%.1f MB)\n", n, size / 1e6);
    printf("Threads per block: %d\n", threadsPerBlock);
    printf("Blocks: %d\n", blocks);
    printf("Kernel time: %.3f ms\n", ms);
    printf("Throughput: %.2f GB/s\n", (3.0 * size) / (ms / 1000) / 1e9);
    // 3x because: read A + read B + write C = 3 memory operations

    // Cleanup
    cudaFree(a); cudaFree(b); cudaFree(c);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}


Writing timing.cu


In [13]:
!nvcc timing.cu -o timing && ./timing


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
c[0] = 3.0 (expected 3.0)
c[9999999] = 3.0 (expected 3.0)

Vector size: 10000000 elements (40.0 MB)
Threads per block: 256
Blocks: 39063
Kernel time: 63.712 ms
Throughput: 1.88 GB/s


In [1]:
%%writefile matmul_naive.cu
#include <stdio.h>
#include <stdlib.h>

// Each thread computes ONE element of C
__global__ void matmul(float *A, float *B, float *C, int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;  // which row of C
    int col = blockIdx.x * blockDim.x + threadIdx.x;  // which col of C

    if (row < M && col < N) {
        float sum = 0.0f;
        for (int i = 0; i < K; i++) {
            sum += A[row * K + i] * B[i * N + col];
        }
        C[row * N + col] = sum;
    }
}

int main() {
    // C = A × B
    // A is M×K, B is K×N, C is M×N
    int M = 512, K = 512, N = 512;

    int sizeA = M * K * sizeof(float);
    int sizeB = K * N * sizeof(float);
    int sizeC = M * N * sizeof(float);

    // Allocate host memory
    float *h_A = (float*)malloc(sizeA);
    float *h_B = (float*)malloc(sizeB);
    float *h_C = (float*)malloc(sizeC);

    // Initialize with random values
    for (int i = 0; i < M * K; i++) h_A[i] = (float)(rand() % 10);
    for (int i = 0; i < K * N; i++) h_B[i] = (float)(rand() % 10);

    // Allocate device memory
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, sizeA);
    cudaMalloc(&d_B, sizeB);
    cudaMalloc(&d_C, sizeC);

    // Copy to GPU
    cudaMemcpy(d_A, h_A, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, sizeB, cudaMemcpyHostToDevice);

    // Launch kernel — 2D grid!
    dim3 threads(16, 16);  // 16×16 = 256 threads per block
    dim3 blocks((N + 15) / 16, (M + 15) / 16);  // ceiling division for both dims

    // Time it
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matmul<<<blocks, threads>>>(d_A, d_B, d_C, M, K, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);

    // Copy result back
    cudaMemcpy(h_C, d_C, sizeC, cudaMemcpyDeviceToHost);

    // Verify one element: C[0][0] = dot(A[row0], B[col0])
    float check = 0;
    for (int i = 0; i < K; i++) check += h_A[i] * h_B[i * N];
    printf("C[0][0] = %.1f (GPU) vs %.1f (CPU check)\n", h_C[0], check);
    printf("\nMatrix: %d × %d × %d\n", M, K, N);
    printf("Kernel time: %.3f ms\n", ms);
    printf("GFLOPS: %.2f\n", (2.0 * M * N * K) / (ms / 1000) / 1e9);

    // Cleanup
    free(h_A); free(h_B); free(h_C);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    return 0;
}


Writing matmul_naive.cu


In [2]:
!nvcc matmul_naive.cu -o matmul_naive && ./matmul_naive


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
C[0][0] = 11057.0 (GPU) vs 11057.0 (CPU check)

Matrix: 512 × 512 × 512
Kernel time: 99.665 ms
GFLOPS: 2.69


In [1]:
%%writefile matmul_bottleneck.cu
#include <stdio.h>
#include <stdlib.h>

// Version 1: Normal (reads from global memory every time)
__global__ void matmul_naive(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int i = 0; i < N; i++) {
            sum += A[row * N + i] * B[i * N + col];
        }
        C[row * N + col] = sum;
    }
}

// Version 2: Fake kernel — same math ops but reads from REGISTERS (no memory)
__global__ void matmul_fake(float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        float a = 1.0f, b = 1.0f;  // fake values in registers
        for (int i = 0; i < N; i++) {
            sum += a * b;  // same number of multiplies, but NO memory reads
        }
        C[row * N + col] = sum;
    }
}

int main() {
    int N = 512;
    int size = N * N * sizeof(float);

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    // Initialize A and B
    float *h_A = (float*)malloc(size);
    for (int i = 0; i < N*N; i++) h_A[i] = 1.0f;
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_A, size, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks((N+15)/16, (N+15)/16);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms;

    // Time naive (real memory reads)
    cudaEventRecord(start);
    matmul_naive<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("Naive (global memory reads): %.3f ms\n", ms);

    // Time fake (no memory reads, just math)
    cudaEventRecord(start);
    matmul_fake<<<blocks, threads>>>(d_C, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("Fake  (registers only):      %.3f ms\n", ms);

    printf("\nThe difference = time spent WAITING for global memory!\n");
    printf("This is why shared memory (tiling) makes it faster.\n");

    free(h_A);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    return 0;
}


Writing matmul_bottleneck.cu


In [2]:
!nvcc matmul_bottleneck.cu -o matmul_bottleneck && ./matmul_bottleneck


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Naive (global memory reads): 114.780 ms
Fake  (registers only):      0.111 ms

The difference = time spent WAITING for global memory!
This is why shared memory (tiling) makes it faster.


In [3]:
%%writefile shared_memory_demo.cu
#include <stdio.h>

#define TILE_SIZE 16

// Tiled matmul using shared memory
__global__ void matmul_tiled(float *A, float *B, float *C, int N) {
    // Shared memory — fast, shared by all threads in this block
    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;
    float sum = 0.0f;

    // Loop over tiles
    for (int t = 0; t < (N + TILE_SIZE - 1) / TILE_SIZE; t++) {

        // Each thread loads ONE element into shared memory
        if (row < N && (t * TILE_SIZE + threadIdx.x) < N)
            tileA[threadIdx.y][threadIdx.x] = A[row * N + t * TILE_SIZE + threadIdx.x];
        else
            tileA[threadIdx.y][threadIdx.x] = 0.0f;

        if (col < N && (t * TILE_SIZE + threadIdx.y) < N)
            tileB[threadIdx.y][threadIdx.x] = B[(t * TILE_SIZE + threadIdx.y) * N + col];
        else
            tileB[threadIdx.y][threadIdx.x] = 0.0f;

        // Wait for ALL threads to finish loading
        __syncthreads();

        // Now compute using fast shared memory (not slow global!)
        for (int i = 0; i < TILE_SIZE; i++) {
            sum += tileA[threadIdx.y][i] * tileB[i][threadIdx.x];
        }

        // Wait before loading next tile
        __syncthreads();
    }

    if (row < N && col < N) {
        C[row * N + col] = sum;
    }
}

// Naive for comparison
__global__ void matmul_naive(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int i = 0; i < N; i++) {
            sum += A[row * N + i] * B[i * N + col];
        }
        C[row * N + col] = sum;
    }
}

int main() {
    int N = 512;
    int size = N * N * sizeof(float);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C1 = (float*)malloc(size);
    float *h_C2 = (float*)malloc(size);

    for (int i = 0; i < N*N; i++) { h_A[i] = (float)(rand()%10); h_B[i] = (float)(rand()%10); }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size); cudaMalloc(&d_B, size); cudaMalloc(&d_C, size);
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks((N+15)/16, (N+15)/16);

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    float ms;

    // Naive
    cudaEventRecord(start);
    matmul_naive<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    cudaMemcpy(h_C1, d_C, size, cudaMemcpyDeviceToHost);
    printf("Naive:  %.3f ms  (%.2f GFLOPS)\n", ms, (2.0*N*N*N)/(ms/1000)/1e9);

    // Tiled
    cudaEventRecord(start);
    matmul_tiled<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    cudaMemcpy(h_C2, d_C, size, cudaMemcpyDeviceToHost);
    printf("Tiled:  %.3f ms  (%.2f GFLOPS)\n", ms, (2.0*N*N*N)/(ms/1000)/1e9);

    // Verify results match
    float maxErr = 0;
    for (int i = 0; i < N*N; i++) {
        float err = fabs(h_C1[i] - h_C2[i]);
        if (err > maxErr) maxErr = err;
    }
    printf("\nMax error between naive and tiled: %.6f\n", maxErr);
    printf("Speedup: %.2fx\n", 0); // will calculate from output

    free(h_A); free(h_B); free(h_C1); free(h_C2);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    return 0;
}


Writing shared_memory_demo.cu


In [4]:
!nvcc shared_memory_demo.cu -o shared_memory_demo && ./shared_memory_demo


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
shared_memory_demo.cu: In function ‘int main()’:
shared_memory_demo.cu:105:8: warning: format ‘%f’ expects argument of type ‘double’, but argument 2 has type ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
  105 |     printf("Speedup: %.2fx\n", 0); // will calculate from output
      |        ^~~~~~~~~~~~~~~~~~  ~
      |                            |
      |                            int
Naive:  30.097 ms  (8.92 GFLOPS)
Tiled:  0.791 ms  (339.52 GFLOPS)

Max error between naive and tiled: 0.000000
Speedup: 0.00x


In [1]:
%%writefile tiled_explained.cu
#include <stdio.h>

#define TILE_SIZE 2  // tiny tile so we can see every step

__global__ void matmul_tiled_verbose(float *A, float *B, float *C, int N) {
    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;
    float sum = 0.0f;

    int numTiles = (N + TILE_SIZE - 1) / TILE_SIZE;

    for (int t = 0; t < numTiles; t++) {

        // Load tile from global → shared
        int aCol = t * TILE_SIZE + threadIdx.x;
        int bRow = t * TILE_SIZE + threadIdx.y;

        if (row < N && aCol < N)
            tileA[threadIdx.y][threadIdx.x] = A[row * N + aCol];
        else
            tileA[threadIdx.y][threadIdx.x] = 0.0f;

        if (bRow < N && col < N)
            tileB[threadIdx.y][threadIdx.x] = B[bRow * N + col];
        else
            tileB[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        // Print what's in shared memory (only block 0,0)
        if (blockIdx.x == 0 && blockIdx.y == 0 && threadIdx.x == 0 && threadIdx.y == 0) {
            printf("\n--- Tile %d loaded into shared memory ---\n", t);
            printf("tileA:              tileB:\n");
            for (int r = 0; r < TILE_SIZE; r++) {
                printf("  [%.0f, %.0f]          [%.0f, %.0f]\n",
                       tileA[r][0], tileA[r][1], tileB[r][0], tileB[r][1]);
            }
        }
        __syncthreads();

        // Compute using shared memory
        for (int i = 0; i < TILE_SIZE; i++) {
            sum += tileA[threadIdx.y][i] * tileB[i][threadIdx.x];
        }

        __syncthreads();
    }

    if (row < N && col < N) {
        C[row * N + col] = sum;
    }
}

int main() {
    // Small 4×4 matrices so we can trace everything
    int N = 4;
    int size = N * N * sizeof(float);

    // A = [[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]]
    float h_A[] = {1,2,3,4, 5,6,7,8, 9,10,11,12, 13,14,15,16};
    // B = [[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1]]  (identity matrix)
    float h_B[] = {1,0,0,0, 0,1,0,0, 0,0,1,0, 0,0,0,1};
    float h_C[16];

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size); cudaMalloc(&d_B, size); cudaMalloc(&d_C, size);
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 threads(TILE_SIZE, TILE_SIZE);  // 2×2 threads per block
    dim3 blocks((N+TILE_SIZE-1)/TILE_SIZE, (N+TILE_SIZE-1)/TILE_SIZE);  // 2×2 blocks

    printf("Matrix A (4x4):\n");
    for (int i = 0; i < 4; i++) {
        printf("  [%.0f, %.0f, %.0f, %.0f]\n", h_A[i*4], h_A[i*4+1], h_A[i*4+2], h_A[i*4+3]);
    }
    printf("\nMatrix B (4x4) = Identity:\n");
    for (int i = 0; i < 4; i++) {
        printf("  [%.0f, %.0f, %.0f, %.0f]\n", h_B[i*4], h_B[i*4+1], h_B[i*4+2], h_B[i*4+3]);
    }
    printf("\nTILE_SIZE = %d, so we process 2x2 chunks at a time", TILE_SIZE);
    printf("\nBlocks: %d×%d, Threads per block: %d×%d\n",
           (N+TILE_SIZE-1)/TILE_SIZE, (N+TILE_SIZE-1)/TILE_SIZE, TILE_SIZE, TILE_SIZE);

    matmul_tiled_verbose<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("\n\nResult C = A × Identity = A:\n");
    for (int i = 0; i < 4; i++) {
        printf("  [%.0f, %.0f, %.0f, %.0f]\n", h_C[i*4], h_C[i*4+1], h_C[i*4+2], h_C[i*4+3]);
    }

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    return 0;
}


Writing tiled_explained.cu


In [2]:
!nvcc tiled_explained.cu -o tiled_explained && ./tiled_explained


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Matrix A (4x4):
  [1, 2, 3, 4]
  [5, 6, 7, 8]
  [9, 10, 11, 12]
  [13, 14, 15, 16]

Matrix B (4x4) = Identity:
  [1, 0, 0, 0]
  [0, 1, 0, 0]
  [0, 0, 1, 0]
  [0, 0, 0, 1]

TILE_SIZE = 2, so we process 2x2 chunks at a time
Blocks: 2×2, Threads per block: 2×2

--- Tile 0 loaded into shared memory ---
tileA:              tileB:
  [1, 2]          [1, 0]
  [5, 6]          [0, 1]

--- Tile 1 loaded into shared memory ---
tileA:              tileB:
  [3, 4]          [0, 0]
  [7, 8]          [0, 0]


Result C = A × Identity = A:
  [1, 2, 3, 4]
  [5, 6, 7, 8]
  [9, 10, 11, 12]
  [13, 14, 15, 16]


In [3]:
%%writefile matmul_benchmark.cu
#include <stdio.h>
#include <stdlib.h>

#define TILE_SIZE 16

__global__ void matmul_naive(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int i = 0; i < N; i++)
            sum += A[row * N + i] * B[i * N + col];
        C[row * N + col] = sum;
    }
}

__global__ void matmul_tiled(float *A, float *B, float *C, int N) {
    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;
    float sum = 0.0f;

    for (int t = 0; t < (N + TILE_SIZE - 1) / TILE_SIZE; t++) {
        int aCol = t * TILE_SIZE + threadIdx.x;
        int bRow = t * TILE_SIZE + threadIdx.y;

        tileA[threadIdx.y][threadIdx.x] = (row < N && aCol < N) ? A[row * N + aCol] : 0.0f;
        tileB[threadIdx.y][threadIdx.x] = (bRow < N && col < N) ? B[bRow * N + col] : 0.0f;

        __syncthreads();
        for (int i = 0; i < TILE_SIZE; i++)
            sum += tileA[threadIdx.y][i] * tileB[i][threadIdx.x];
        __syncthreads();
    }

    if (row < N && col < N)
        C[row * N + col] = sum;
}

float benchmark(void (*kernel)(float*, float*, float*, int), float *d_A, float *d_B, float *d_C, int N) {
    dim3 threads(16, 16);
    dim3 blocks((N+15)/16, (N+15)/16);

    // Warmup
    kernel<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    for (int i = 0; i < 5; i++)
        kernel<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    return ms / 5.0f;
}

int main() {
    int sizes[] = {128, 256, 512, 1024, 2048};
    int numSizes = 5;

    printf("┌──────────┬────────────┬────────────┬──────────┬──────────┬─────────┐\n");
    printf("│  Size    │ Naive (ms) │ Tiled (ms) │ Naive GF │ Tiled GF │ Speedup │\n");
    printf("├──────────┼────────────┼────────────┼──────────┼──────────┼─────────┤\n");

    for (int s = 0; s < numSizes; s++) {
        int N = sizes[s];
        int size = N * N * sizeof(float);

        float *h_A = (float*)malloc(size);
        for (int i = 0; i < N*N; i++) h_A[i] = (float)(rand() % 10);

        float *d_A, *d_B, *d_C;
        cudaMalloc(&d_A, size); cudaMalloc(&d_B, size); cudaMalloc(&d_C, size);
        cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, h_A, size, cudaMemcpyHostToDevice);

        float naive_ms = benchmark(matmul_naive, d_A, d_B, d_C, N);
        float tiled_ms = benchmark(matmul_tiled, d_A, d_B, d_C, N);

        double flops = 2.0 * N * N * N;
        float naive_gf = flops / (naive_ms / 1000) / 1e9;
        float tiled_gf = flops / (tiled_ms / 1000) / 1e9;
        float speedup = naive_ms / tiled_ms;

        printf("│ %4d×%-4d│   %7.3f  │   %7.3f  │  %6.1f  │  %6.1f  │  %5.1fx │\n",
               N, N, naive_ms, tiled_ms, naive_gf, tiled_gf, speedup);

        free(h_A);
        cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    }

    printf("└──────────┴────────────┴────────────┴──────────┴──────────┴─────────┘\n");
    printf("\nT4 peak: ~8,100 GFLOPS (FP32)\n");
    return 0;
}


Writing matmul_benchmark.cu


In [4]:
!nvcc matmul_benchmark.cu -o matmul_benchmark && ./matmul_benchmark


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
┌──────────┬────────────┬────────────┬──────────┬──────────┬─────────┐
│  Size    │ Naive (ms) │ Tiled (ms) │ Naive GF │ Tiled GF │ Speedup │
├──────────┼────────────┼────────────┼──────────┼──────────┼─────────┤
│  128×128 │     0.031  │     0.021  │   134.7  │   198.6  │    1.5x │
│  256×256 │     0.156  │     0.107  │   214.8  │   313.1  │    1.5x │
│  512×512 │     1.147  │     0.744  │   234.1  │   361.0  │    1.5x │
│ 1024×1024│     6.666  │     4.202  │   322.1  │   511.1  │    1.6x │
│ 2048×2048│    48.437  │    22.073  │   354.7  │   778.3  │    2.2x │
└──────────┴────────────┴────────────┴──────────┴──────────┴─────────┘

T4 peak: ~8,100 GFLOPS (FP32)


In [ ]:
%%writefile coalescing.cu
#include <stdio.h>

#define N 4096
#define BLOCK_SIZE 256

// GOOD: Consecutive threads read consecutive memory addresses (coalesced)
__global__ void coalesced_read(float *data, float *out) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N * N) {
        out[i] = data[i] * 2.0f;  // thread 0 reads [0], thread 1 reads [1], ...
    }
}

// BAD: Consecutive threads read with stride (non-coalesced)
__global__ void strided_read(float *data, float *out) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N * N) {
        // Thread 0 reads [0], thread 1 reads [N], thread 2 reads [2N]...
        int row = i % N;
        int col = i / N;
        out[i] = data[row * N + col] * 2.0f;  // column-major access = strided!
    }
}

int main() {
    int size = N * N * sizeof(float);
    float *d_data, *d_out;
    cudaMalloc(&d_data, size);
    cudaMalloc(&d_out, size);

    // Initialize
    float *h_data = (float*)malloc(size);
    for (int i = 0; i < N*N; i++) h_data[i] = 1.0f;
    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);

    int blocks = (N * N + BLOCK_SIZE - 1) / BLOCK_SIZE;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms;

    // Warmup
    coalesced_read<<<blocks, BLOCK_SIZE>>>(d_data, d_out);
    strided_read<<<blocks, BLOCK_SIZE>>>(d_data, d_out);
    cudaDeviceSynchronize();

    // Benchmark coalesced
    cudaEventRecord(start);
    for (int i = 0; i < 10; i++)
        coalesced_read<<<blocks, BLOCK_SIZE>>>(d_data, d_out);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("Coalesced (consecutive):  %.3f ms  (%.1f GB/s)\n",
           ms/10, (2.0*size/1e9)/(ms/10/1000));

    // Benchmark strided
    cudaEventRecord(start);
    for (int i = 0; i < 10; i++)
        strided_read<<<blocks, BLOCK_SIZE>>>(d_data, d_out);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("Strided (column-major):   %.3f ms  (%.1f GB/s)\n",
           ms/10, (2.0*size/1e9)/(ms/10/1000));

    printf("\n");
    printf("WHY coalesced is faster:\n");
    printf("─────────────────────────\n");
    printf("GPU reads memory in 128-byte chunks (cache lines).\n");
    printf("If 32 threads in a warp read consecutive addresses,\n");
    printf("ONE 128-byte read serves ALL 32 threads.\n");
    printf("\n");
    printf("If threads read scattered addresses,\n");
    printf("GPU needs MULTIPLE reads to serve the same 32 threads.\n");

    free(h_data);
    cudaFree(d_data); cudaFree(d_out);
    return 0;
}
